# SQL in Spark - Review
- Notebook by Adam Lang
- Date: 9-1-2026

## Overview
- We will go over:
1. SQL using Python API in spark
2. SPARK SQL
3. JOINS

## 1. SQL using Python API

In [0]:
## this uses the python API
# databricksdemo.default.movies
df_marvel = spark.sql("select * from databricksdemo.default.movies where studio='Marvel Studios'")
df_marvel.show(truncate=False)

+-------------------------------------------+---------+------------+-----------+--------------+------+-------+--------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio        |budget|revenue|unit    |currency|language|
+-------------------------------------------+---------+------------+-----------+--------------+------+-------+--------+--------+--------+
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios|200.0 |954.8  |Millions|USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios|165.0 |644.8  |Millions|USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel Studios|180.0 |854.0  |Millions|USD     |English |
|Thor: Love and Thunder                     |Hollywood|2022        |6.8        |Marvel Studios|250.0 |670.0  |Millions|USD     |English |
|Avengers: Endgame                

## 2. SQL using `%sql`
- direct sql query
- This result is stored as `_sqldf` and can be used in other Python and SQL cells.

In [0]:
%sql
select * from databricksdemo.default.movies where studio='Marvel Studios'

title,industry,release_year,imdb_rating,studio,budget,revenue,unit,currency,language
Doctor Strange in the Multiverse of Madness,Hollywood,2022,7,Marvel Studios,200.0,954.8,Millions,USD,English
Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,165.0,644.8,Millions,USD,English
Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,180.0,854.0,Millions,USD,English
Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,250.0,670.0,Millions,USD,English
Avengers: Endgame,Hollywood,2019,8.4,Marvel Studios,400.0,2798.0,Millions,USD,English
Avengers: Infinity War,Hollywood,2018,8.4,Marvel Studios,400.0,2048.0,Millions,USD,English
Captain America: The First Avenger,Hollywood,2011,6.9,Marvel Studios,216.7,370.6,Millions,USD,English
Captain America: The Winter Soldier,Hollywood,2014,7.8,Marvel Studios,177.0,714.4,Millions,USD,English


# Create DataFrame --> Run SQL 
- We will create a temporary synthetic dataframe

In [0]:
from pyspark.sql import functions as F, types as T

data = [
    ("2017-01-01", 32.0,  6.0,  "Rain"),
    ("2017-01-04", None,  9.0,  "Sunny"),
    ("2017-01-05", 28.0,  None, "Snow"),
    ("2017-01-06", None,  7.0,  None),
    ("2017-01-07", 32.0,  None, "Rain"),
    ("2017-01-08", None,  None, "Sunny"),
    ("2017-01-09", None,  None, None),
    ("2017-01-10", 34.1,  8.1,  "Cloudy"),
    ("2017-01-11", 40.0, 12.0,  "Sunny"),
]

## schema
schema = "day string, temperature double, windspeed double, event string"
df = spark.createDataFrame(data, schema)
df = df.withColumn("day", F.to_date("day", "yyyy-MM-dd"))  # normalize to DateType

display(df)

day,temperature,windspeed,event
2017-01-01,32.0,6.0,Rain
2017-01-04,null,9.0,Sunny
2017-01-05,28.0,null,Snow
2017-01-06,null,7.0,null
2017-01-07,32.0,null,Rain
2017-01-08,null,null,Sunny
2017-01-09,null,null,null
2017-01-10,34.1,8.1,Cloudy
2017-01-11,40.0,12.0,Sunny


## Run SQL queries
- we can create a temporary view to do this

In [0]:
## create temp view to run SQL query
df.createOrReplaceTempView("weather") ## -- this is SESSION scoped (per session)
# df.createGlobalTempView("global_weather") ## -- this is GLOBAL scoped per cluster (not suported on serverless)

In [0]:
%sql
-- sql query to run 
SELECT event, ROUND(AVG(temperature), 1) AS avg_temp
FROM weather -- VIEW created above
GROUP BY event 
ORDER BY avg_temp DESC;

event,avg_temp
Sunny,40.0
Cloudy,34.1
Rain,32.0
Snow,28.0
null,null


In [0]:
## can also use spark.sql direct in python
spark.sql("""
SELECT day, temperature, windspeed, event
FROM weather
WHERE temperature IS NOT NULL
ORDER BY day
""").show()

+----------+-----------+---------+------+
|       day|temperature|windspeed| event|
+----------+-----------+---------+------+
|2017-01-01|       32.0|      6.0|  Rain|
|2017-01-05|       28.0|     NULL|  Snow|
|2017-01-07|       32.0|     NULL|  Rain|
|2017-01-10|       34.1|      8.1|Cloudy|
|2017-01-11|       40.0|     12.0| Sunny|
+----------+-----------+---------+------+



# JOINS

In [0]:
## imports
from pyspark.sql import functions as F, types as T

## Create synthetic df #1
rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

## Create synthetic df #2
rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

## create customers schema
schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

## create orders schema
schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])

## create spark df's
df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders    = spark.createDataFrame(rows_orders,    schema_orders)

## display
display(df_customers)
display(df_orders)

customer_id,name,country,vip
1,Asha,IN,true
2,Bob,US,false
3,Chen,CN,true
4,Diana,US,null
null,Ghost,UK,false


order_id,customer_id,amount,country
101,1,120.0,IN
102,1,80.0,IN
103,2,50.0,US
104,5,30.0,DE
105,3,200.0,CN
106,null,15.0,UK
107,3,40.0,CN
108,2,75.0,US


## Join on a column

### Inner Join
- Only returns rows that have matching values in both tables.  

In [0]:
## we can join on a column similar to python pandas
df_inner = df_orders.join(df_customers,
               on="customer_id",
               how="inner") # default is inner join 


## display
display(df_inner)

customer_id,order_id,amount,country,name,country,vip
1,101,120.0,IN,Asha,IN,true
1,102,80.0,IN,Asha,IN,true
2,103,50.0,US,Bob,US,false
3,105,200.0,CN,Chen,CN,true
3,107,40.0,CN,Chen,CN,true
2,108,75.0,US,Bob,US,false


## Left Join
- Returns all rows from the left table, and only the matched rows from the right table

In [0]:
## left join example
df_left = df_orders.join(df_customers,
                         on="customer_id",
                         how="left") # default is left join

## display
display(df_left)

customer_id,order_id,amount,country,name,country,vip
1,101,120.0,IN,Asha,IN,true
1,102,80.0,IN,Asha,IN,true
2,103,50.0,US,Bob,US,false
5,104,30.0,DE,null,null,null
3,105,200.0,CN,Chen,CN,true
null,106,15.0,UK,null,null,null
3,107,40.0,CN,Chen,CN,true
2,108,75.0,US,Bob,US,false


## SQL Alias

In [0]:

o = df_orders.alias("o")
o.show()

+--------+-----------+------+-------+
|order_id|customer_id|amount|country|
+--------+-----------+------+-------+
|     101|          1| 120.0|     IN|
|     102|          1|  80.0|     IN|
|     103|          2|  50.0|     US|
|     104|          5|  30.0|     DE|
|     105|          3| 200.0|     CN|
|     106|       NULL|  15.0|     UK|
|     107|          3|  40.0|     CN|
|     108|          2|  75.0|     US|
+--------+-----------+------+-------+



In [0]:
## now create alias for both customers and orders tables
o, c = df_orders.alias("o"), df_customers.alias("c")

## inner join
df_inner = o.join(c, on="customer_id", how="inner")
display(df_inner)

customer_id,order_id,amount,country,name,country,vip
1,101,120.0,IN,Asha,IN,true
1,102,80.0,IN,Asha,IN,true
2,103,50.0,US,Bob,US,false
3,105,200.0,CN,Chen,CN,true
3,107,40.0,CN,Chen,CN,true
2,108,75.0,US,Bob,US,false


## Alias for columns 
- This is useful when you want to link a column with the same name to its original table, such as "country" above.

In [0]:
## create alias cols -- inner join
df_inner_clean = (
    o.join(c, on="customer_id", how="inner")
    .select("order_id", "customer_id", "amount",
            F.col("o.country").alias("ship_country"),
            "name",
            F.col("c.country").alias("cust_country"),
            "vip")
)
display(df_inner_clean)

order_id,customer_id,amount,ship_country,name,cust_country,vip
101,1,120.0,IN,Asha,IN,true
102,1,80.0,IN,Asha,IN,true
103,2,50.0,US,Bob,US,false
105,3,200.0,CN,Chen,CN,true
107,3,40.0,CN,Chen,CN,true
108,2,75.0,US,Bob,US,false


## Full (outer) JOIN
- similar to a union between two sets
- returns all rows where there is a match in wither left or right tables.

In [0]:
## full join
display(o.join(c, on="customer_id", how="full"))

customer_id,order_id,amount,country,name,country,vip
1,101,120.0,IN,Asha,IN,true
1,102,80.0,IN,Asha,IN,true
2,103,50.0,US,Bob,US,false
5,104,30.0,DE,null,null,null
3,105,200.0,CN,Chen,CN,true
null,106,15.0,UK,null,null,null
3,107,40.0,CN,Chen,CN,true
2,108,75.0,US,Bob,US,false
null,null,null,null,Ghost,UK,false
4,null,null,null,Diana,US,null


## Left-Semi Join
- A LEFT SEMI JOIN returns only the rows from the left-hand table that have at least one matching record in the right-hand table.
- Unlike a standard INNER JOIN or LEFT JOIN, it never returns columns from the right table and never duplicates rows from the left table if multiple matches exist on the right side. 
- It essentially acts as an existential filter

In [0]:
## left-semi join
display(o.join(c, on="customer_id", how="left_semi")) ## orders with a known customer

customer_id,order_id,amount,country
1,101,120.0,IN
1,102,80.0,IN
2,103,50.0,US
3,105,200.0,CN
3,107,40.0,CN
2,108,75.0,US


## Left-anti join
- A left anti join returns only the rows from the left table that have no matching rows in the right table.
- How It Works:
    - The left table acts as the main data source.
    - The right table acts as a check or filter.
    - If a match is found between the two tables, that row is removed.
    - If no match is found, the row from the left table is kept

- Common Use Cases
    - Finding customers who have never placed an order.
    - Locating users who have not visited a website.
    - Finding items in an inventory that are missing from a sales list.

- Below we can use it to find orphan orders with no matching customers.

In [0]:
display(o.join(c, on="customer_id", how="left_anti")) ## orphan orders with no matching customers

customer_id,order_id,amount,country
5,104,30.0,DE
null,106,15.0,UK


## Composite Join (multi-key)
- We can join with multiple keys from the dataset

In [0]:
df_multi = o.join(c,
                  on=["customer_id", "country"],
                  how="inner")
display(df_multi)

customer_id,country,order_id,amount,name,vip
1,IN,101,120.0,Asha,true
1,IN,102,80.0,Asha,true
2,US,103,50.0,Bob,false
3,CN,105,200.0,Chen,true
3,CN,107,40.0,Chen,true
2,US,108,75.0,Bob,false
